# 02 - Agent Execution Loop Lab
This lab explores how an agent actually executes. It progresses from a naive, dangerous `while` loop to a fully typed, bounded execution loop mimicking real enterprise constraints.

## Part 1 — Naive Agent Loop
An unbounded loop with no step limit and no error classification.

In [ ]:
def naive_decide(state):
    if 'error' in str(state.get('last_obs')):
        return 'retry_same_action'
    return 'final_answer'

state = {'last_obs': 'API Error: Timeout'}
steps = 0
print('Starting naive loop (simulating failure)...')
while state.get('last_obs') != 'success':
    action = naive_decide(state)
    # Simulate execution
    state['last_obs'] = 'API Error: Timeout'
    steps += 1
    if steps > 5:
        print('DANGER: Runaway loop detected. Forced exit.')
        break

## Part 2 — Typed State
Introduce Pydantic to track exactly what the agent is doing.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class AgentState(BaseModel):
    goal: str
    evidence: list[str] = Field(default_factory=list)
    steps: int = 0
    tool_calls: int = 0
    retries: int = 0
    seen_actions: set[str] = Field(default_factory=set)
    terminal_reason: Optional[str] = None

state = AgentState(goal='Diagnose checkout incident in EU')
print('Typed state initialized:', state.model_dump_json(indent=2))

## Part 3 — Typed Actions
Define structured tool decisions and show validation failure.

In [ ]:
from typing import Literal, Dict, Any
from pydantic import ValidationError

class ToolCall(BaseModel):
    tool: Literal['get_service_health', 'search_incidents', 'query_checkout_logs']
    arguments: Dict[str, Any]

try:
    # Simulated bad proposal from LLM
    bad_call = ToolCall(tool='restart_server', arguments={})
except ValidationError as e:
    print('Runtime caught invalid tool proposal:')
    print(e)

## Part 4 — Realistic Tool Registry
Northstar read-only tools using local deterministic fixtures.

In [ ]:
def get_service_health(region: str) -> str:
    if region == 'EU': return 'Status: Degraded (Checkout DB High Latency)'
    return 'Status: Healthy'

def query_checkout_logs(region: str) -> str:
    return f'[{region}] 45 timeouts matching payment gateway connection timeout.'

def get_runbook(topic: str) -> str:
    if 'timeout' in topic.lower(): return 'Runbook: Check recent deployments. If recent deploy, consider rollback.'
    return 'Runbook not found.'

def get_recent_deployments(service: str) -> str:
    return 'Deployment v1.14 shipped 20 minutes ago to EU checkout service.'

print('Tool registry initialized.')

## Part 5 & 6 — Bounded Execution Loop & Error Classification
Executing with limits, duplicate action detection, and explicit terminal reasons.

In [ ]:
def dispatch_tool(tool_name: str, args: dict) -> str:
    if tool_name == 'get_service_health': return get_service_health(**args)
    if tool_name == 'query_checkout_logs': return query_checkout_logs(**args)
    if tool_name == 'get_runbook': return get_runbook(**args)
    if tool_name == 'get_recent_deployments': return get_recent_deployments(**args)
    if tool_name == 'simulate_timeout': return 'HTTP 504 Timeout'
    if tool_name == 'simulate_denied': return 'Permission Denied: write access required'
    return 'Unknown tool'

def run_bounded_loop(mock_decisions):
    loop_state = AgentState(goal='Diagnose EU checkout')
    MAX_STEPS = 5
    
    for step, decision in enumerate(mock_decisions):
        if loop_state.steps >= MAX_STEPS:
            loop_state.terminal_reason = 'STEP_BUDGET_EXHAUSTED'
            break
            
        print(f'\nStep {step + 1}: Model proposes {decision['tool']}({decision['arguments']})')
        
        # Fingerprint for no-progress
        fingerprint = f"{decision['tool']}:{str(decision['arguments'])}"
        if fingerprint in loop_state.seen_actions:
            print('NO_PROGRESS: Repeated action detected. Escalating.')
            loop_state.terminal_reason = 'NO_PROGRESS'
            break
        loop_state.seen_actions.add(fingerprint)
        
        # Error Classification
        obs = dispatch_tool(decision['tool'], decision['arguments'])
        print(f'Observation: {obs}')
        
        if 'Permission Denied' in obs:
            print('FATAL ERROR: Policy block. Cannot retry.')
            loop_state.terminal_reason = 'POLICY_BLOCK'
            break
            
        if 'Timeout' in obs:
            print('TRANSIENT ERROR: Retrying... (simulate backoff)')
            loop_state.retries += 1
            loop_state.seen_actions.remove(fingerprint) # Allow retry
            continue
            
        loop_state.evidence.append(obs)
        loop_state.steps += 1
        loop_state.tool_calls += 1
        
    if not loop_state.terminal_reason:
        loop_state.terminal_reason = 'SUCCESS'
        
    print(f'\nLoop terminated with reason: {loop_state.terminal_reason}')
    return loop_state

print('Bounded loop runtime defined.')

## Part 7 — No-Progress Loop Detection
Deliberately send the same action twice to verify the harness stops it.

In [ ]:
mock_no_progress = [
    {'tool': 'get_service_health', 'arguments': {'region': 'EU'}},
    {'tool': 'get_service_health', 'arguments': {'region': 'EU'}} # Duplicate
]
final_state = run_bounded_loop(mock_no_progress)

## Part 8 — Dynamic Replanning
Evidence-triggered replanning: a new deployment invalidates the initial timeout hypothesis.

In [ ]:
mock_replanning = [
    {'tool': 'query_checkout_logs', 'arguments': {'region': 'EU'}},
    {'tool': 'get_recent_deployments', 'arguments': {'service': 'checkout'}}
]
print('Simulating Replanning...\nIf recent deployments is true, state changes to checking rollback procedures.')
final_state = run_bounded_loop(mock_replanning)
if any('Deployment v1.14' in e for e in final_state.evidence):
    print('\nREPLAN TRIGGERED: New deployment evidence found. Discarding initial hypothesis.')

## Part 9 — Plan-and-Execute
Using a coarse plan instead of step-by-step ReAct.

In [ ]:
plan = [
    {'tool': 'get_service_health', 'arguments': {'region': 'EU'}},
    {'tool': 'get_recent_deployments', 'arguments': {'service': 'checkout'}}
]
print('Executing fixed plan...')
run_bounded_loop(plan)

## Part 10 — Reflection
Adding a revision cap to prevent endless loops.

In [ ]:
draft = 'We should restart the production database.'
revisions = 0
MAX_REVISIONS = 1
while revisions < MAX_REVISIONS:
    print(f'Checking draft: {draft}')
    if 'restart' in draft:
        print('Rubric Failed: Cannot propose unauthorized production changes.')
        draft = 'Investigate recent deployments.'
        revisions += 1
    else:
        print('Rubric Passed.')
        break
if revisions == MAX_REVISIONS:
    print('Reflection loop capped.')

## Part 11 — Event-Driven Resume
Idempotency using event IDs.

In [ ]:
processed_events = set()
def on_event(event_id, data):
    if event_id in processed_events:
        print(f'Event {event_id} already processed. Idempotency activated.')
        return
    print(f'Processing new event {event_id}: {data}')
    processed_events.add(event_id)

on_event('EVT_001', 'Customer reply arrived')
on_event('EVT_001', 'Customer reply arrived') # Duplicate delivery simulation

## Part 12 — LangGraph Mapping
How these manual loops map to LangGraph state-machine architectures.

In [ ]:
from typing import TypedDict, Annotated
class GraphState(TypedDict):
    evidence: list[str]
    terminal_reason: str

def decide_node(state: GraphState):
    pass

def tool_node(state: GraphState):
    pass

print('In LangGraph, you would map decide_node -> conditional_edge -> tool_node -> decide_node.')

## Part 13 — Optional Real LLM
Requires OPENAI_API_KEY. (Skipping live API call to ensure determinism).

In [ ]:
import os
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print('API Key found. You could initialize OpenAI client here.')
else:
    print('No API key. Running in deterministic fixture mode only.')

## Part 14 — Evaluation Harness
Run standard scenarios to test terminal condition assertions.

In [ ]:
eval_scenarios = [
    ('Transient Failure', [{'tool': 'simulate_timeout', 'arguments': {}}]),
    ('Permission Denied', [{'tool': 'simulate_denied', 'arguments': {}}]),
]
for name, sequence in eval_scenarios:
    print(f'\n--- Evaluating {name} ---')
    state = run_bounded_loop(sequence)
    assert state.terminal_reason in ['SUCCESS', 'POLICY_BLOCK', 'NO_PROGRESS', 'STEP_BUDGET_EXHAUSTED', 'TOOL_FAILURE'], 'Invariant violated!'